# AuxiliaryASR CTC Entropy Evaluation

This notebook loads a trained AuxiliaryASR model, prepares the validation dataset, and computes summary statistics of the CTC output distribution.

In [ ]:
# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device_name = torch.cuda.get_device_name(i)
        devices.append({
            'type': 'CUDA',
            'available': True,
            'name': device_name,
            'index': i
        })
else:
    devices.append({'type': 'CUDA', 'available': False, 'name': 'N/A'})
devices

In [ ]:
# change folder into the root of the ASR project
import os

if not os.path.isdir('Configs'):
    %cd ../

!pwd

## Imports and helper utilities

In [ ]:
# import packages, define helper utilities
import os
import yaml
import torch
import pandas as pd

from models import ASRCNN
from meldataset import build_dataloader
from token_map import build_token_map_from_data

def cfg_get_nested(cfg: dict, path, default=None, sep='.'):
    """Get a nested value from a dict using a list of keys or a dot-separated string."""
    if isinstance(path, str):
        keys = path.split(sep) if path else []
    else:
        keys = path

    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

def resolve_ssl_frontend_config(config):
    ssl_cfg = cfg_get_nested(config, 'ssl_frontend', {}) or {}
    if not isinstance(ssl_cfg, dict):
        return {'enabled': False}
    resolved = dict(ssl_cfg)
    if not resolved.get('enabled', False):
        return {'enabled': False}

    default_models = {
        'wav2vec2': 'facebook/wav2vec2-base',
        'hubert': 'facebook/hubert-base-ls960',
        'wavlm': 'microsoft/wavlm-base',
        'xlsr': 'facebook/wav2vec2-large-xlsr-53',
    }
    selected_architecture = resolved.get('architecture')
    selected_model_name = resolved.get('pretrained_model_name')

    for candidate in ('wav2vec2', 'hubert', 'wavlm', 'xlsr'):
        candidate_cfg = resolved.get(candidate)
        if isinstance(candidate_cfg, dict) and candidate_cfg.get('enabled'):
            selected_architecture = candidate
            candidate_model = candidate_cfg.get('pretrained_model_name') or candidate_cfg.get('model_name')
            if candidate_model:
                selected_model_name = candidate_model
            break

    if selected_architecture is None:
        selected_architecture = 'wav2vec2'
    if not selected_model_name:
        selected_model_name = default_models.get(selected_architecture)

    if not selected_model_name:
        return {'enabled': False}

    resolved['architecture'] = selected_architecture
    resolved['pretrained_model_name'] = selected_model_name
    resolved['enabled'] = True
    return resolved

def get_preprocess_sample_rate(config):
    if 'preprocess_params' in config:
        key = 'preprocess_params'
    elif 'preprocess_parasm' in config:
        key = 'preprocess_parasm'
    else:
        key = None
    if key is None:
        return 24000
    return cfg_get_nested(config, f"{key}.sr", 24000)

def unpack_ctc_outputs(model_output):
    if isinstance(model_output, dict):
        logits = (
            model_output.get('logits_ctc')
            or model_output.get('ctc_logit')
            or model_output.get('logits')
            or model_output.get('ctc')
        )
        encoder_lengths = model_output.get('encoder_lengths')
    elif isinstance(model_output, (tuple, list)):
        logits = model_output[0] if len(model_output) > 0 else None
        encoder_lengths = None
        for item in model_output[1:]:
            if torch.is_tensor(item) and item.dim() == 1:
                encoder_lengths = item
                break
    else:
        logits = model_output
        encoder_lengths = None
    if logits is None:
        raise ValueError('Could not extract CTC logits from model output.')
    return logits, encoder_lengths

def prepare_batch_for_model(batch, use_ssl_frontend, device):
    if use_ssl_frontend:
        texts, text_lens, mels, mel_lens, wave_tensor, wave_lengths = batch
        features = wave_tensor.to(device)
        feature_lengths = wave_lengths.to(device)
        mel_inputs = mels
        mel_lengths = mel_lens
    else:
        texts, text_lens, mels, mel_lens = batch
        features = mels.to(device)
        feature_lengths = mel_lens.to(device)
        mel_inputs = mels
        mel_lengths = mel_lens
    return {
        'texts': texts,
        'text_lengths': text_lens,
        'mel_inputs': mel_inputs,
        'mel_lengths': mel_lengths,
        'features': features,
        'feature_lengths': feature_lengths,
        'feature_lengths_cpu': feature_lengths.detach().to(torch.long).cpu(),
    }

def load_token_map_from_config(config):
    token_src = config.get('phoneme_maps_path')
    if not token_src:
        return build_token_map_from_data(
            config.get('train_data'),
            config.get('val_data'),
            config.get('ood_data'),
            apply_asr_tokenizer=True,
        )
    if isinstance(token_src, dict):
        return token_src
    csv = pd.read_csv(token_src, header=None).values
    return {word: index for word, index in csv}

def load_asr_model(model_path, config_path, device):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    token_map = load_token_map_from_config(config)

    model_params = cfg_get_nested(config, 'model_params', {
        'input_dim': 80,
        'hidden_dim': 256,
        'n_token': len(token_map),
        'token_embedding_dim': 512,
        'n_layers': 5,
        'location_kernel_size': 31,
    })
    if 'n_token' not in model_params:
        model_params['n_token'] = len(token_map)

    ssl_frontend_config = resolve_ssl_frontend_config(config)
    model = ASRCNN(**model_params, ssl_frontend_config=ssl_frontend_config)

    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    state_dict = checkpoint.get('model', checkpoint)
    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        sanitized_state = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(sanitized_state)

    model.to(device)
    model.eval()
    return model, config, token_map, ssl_frontend_config

def build_dev_dataloader(config, device, batch_size=None, num_workers=None, ssl_enabled=False):
    val_csv_path = config.get('val_data')
    if val_csv_path is None:
        raise ValueError("Validation CSV path ('val_data') not found in the config.")

    with open(val_csv_path, 'r', encoding='utf-8') as f:
        raw_lines = [line.rstrip('
') for line in f]

    path_list = []
    for raw in raw_lines:
        if not raw.strip():
            continue
        parts = raw.split('|')
        if len(parts) == 1:
            continue
        path = parts[0]
        if len(parts) == 2:
            text = parts[1]
            speaker = ''
        else:
            text = '|'.join(parts[1:-1])
            speaker = parts[-1]
        path_list.append([path, text, speaker])

    if batch_size is None:
        batch_size = int(cfg_get_nested(config, 'eval_params.batch_size', cfg_get_nested(config, 'batch_size', 4)))
    if num_workers is None:
        num_workers = int(cfg_get_nested(config, 'dataloader_params.val_num_workers', 2))

    dataset_params = {
        'dict_path': cfg_get_nested(config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested(config, 'preprocess_params.sr', cfg_get_nested(config, 'preprocess_parasm.sr', 24000)),
        'spect_params': cfg_get_nested(
            config,
            'preprocess_params.spect_params',
            {'n_fft': 1024, 'win_length': 1024, 'hop_length': 300},
        ),
        'mel_params': cfg_get_nested(
            config, 'preprocess_params.mel_params', {'n_mels': 80}
        ),
    }

    collate_config = {'return_wave': bool(ssl_enabled)}
    device_flag = device.type if isinstance(device, torch.device) else str(device)
    loader = build_dataloader(
        path_list=path_list,
        validation=True,
        batch_size=batch_size,
        num_workers=num_workers,
        device=device_flag,
        collate_config=collate_config,
        dataset_config=dataset_params,
    )
    return loader


## Load model, configuration, and validation loader

In [ ]:
checkpoint_dir = 'Checkpoint'
config_path = 'Checkpoint/config.yml'

if not os.path.isdir(checkpoint_dir):
    raise FileNotFoundError(f"Checkpoint directory '{checkpoint_dir}' not found.")

ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.startswith('epoch_') and f.endswith('.pth')]
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoint files found in '{checkpoint_dir}'.")

ckpt_files = sorted(ckpt_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
print(f"Loading model from {checkpoint_dir}/{ckpt_files[-1]}")
model_path = os.path.join(checkpoint_dir, ckpt_files[-1])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, config, token_map, ssl_frontend_config = load_asr_model(model_path, config_path, device)
SSL_FRONTEND_ENABLED = bool(ssl_frontend_config.get('enabled', False))
INPUT_SAMPLE_RATE = get_preprocess_sample_rate(config)

print(f'model --> {model_path}')
print(f'config --> {config_path}')
phoneme_source = config.get('phoneme_maps_path', 'built from dataset')
print(f'dictionary --> {phoneme_source}')
if SSL_FRONTEND_ENABLED:
    arch = ssl_frontend_config.get('architecture', 'unknown')
    pretrained_name = ssl_frontend_config.get('pretrained_model_name', 'unknown')
    print(f"SSL frontend enabled: {arch} ({pretrained_name})")
else:
    print('SSL frontend disabled; using mel-spectrogram inputs.')

dev_loader = build_dev_dataloader(config, device, ssl_enabled=SSL_FRONTEND_ENABLED)
print(f'Validation dataset size: {len(dev_loader.dataset)} samples')

if ' ' not in token_map:
    raise KeyError("The vocabulary does not contain the blank symbol ' '.")
BLANK_ID = token_map[' ']
print(f'Blank token id: {BLANK_ID}')


## CTC entropy statistics
- mean_maxprob often ends up > 0.6;
- mean_entropy should drop well below log(V);
- mean_blank_rate typically sits around 0.6–0.8 for good CTCs (too high can indicate over-blanking; too low can indicate smeared posteriors).

In [ ]:
# entropy statistics computation
import math

@torch.no_grad()
def ctc_entropy_stats(model, dev_loader, device=None, blank_id=0):
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    entropies, maxp, blank_rates = [], [], []

    for batch in dev_loader:
        batch_data = prepare_batch_for_model(batch, SSL_FRONTEND_ENABLED, device)
        features = batch_data['features']
        feature_lengths = batch_data['feature_lengths']
        feature_lengths_cpu = batch_data['feature_lengths_cpu']

        model_kwargs = {'input_lengths': feature_lengths}
        if SSL_FRONTEND_ENABLED:
            model_kwargs['sampling_rate'] = INPUT_SAMPLE_RATE

        outputs = model(features, **model_kwargs)
        logits, encoder_lengths = unpack_ctc_outputs(outputs)

        logp = logits.log_softmax(-1).cpu()
        probs = logp.exp()
        entropy = -(probs * logp).sum(-1)
        max_probs, ids = probs.max(-1)
        blank_mask = ids.eq(blank_id)

        if encoder_lengths is None:
            logit_lens = feature_lengths_cpu
        else:
            logit_lens = encoder_lengths.detach().to(torch.long).cpu()

        for h, m, bmask, L in zip(entropy, max_probs, blank_mask, logit_lens):
            length = int(L)
            entropies.append(h[:length].mean().item())
            maxp.append(m[:length].mean().item())
            blank_rates.append(bmask[:length].float().mean().item())

    return {
        'mean_entropy': float(torch.tensor(entropies).mean()),
        'mean_maxprob': float(torch.tensor(maxp).mean()),
        'mean_blank_rate': float(torch.tensor(blank_rates).mean()),
    }


stats = ctc_entropy_stats(model, dev_loader, device=device, blank_id=BLANK_ID)
print(stats)
